# MOIRAI 1.1 multivariate compatibility gate
Run top-to-bottom on a Colab Pro+ GPU. This is a compatibility smoke, not test evaluation or a performance experiment.

In [ ]:
import os
import subprocess
from pathlib import Path

ROOT = Path("/content/tsfm-zero-few-shot-crossover")
URL = "https://github.com/Han-Youseung/tsfm-zero-few-shot-crossover.git"
if ROOT.exists():
    subprocess.run(["git", "-C", str(ROOT), "pull", "--ff-only", "origin", "main"], check=True)
else:
    subprocess.run(["git", "clone", URL, str(ROOT)], check=True)
os.chdir(ROOT)
head = subprocess.check_output(["git", "rev-parse", "HEAD"], text=True).strip()
origin = subprocess.check_output(["git", "rev-parse", "origin/main"], text=True).strip()
assert head == origin, (head, origin)
print("project commit", head)

In [ ]:
%pip install -q -r requirements/moirai1.txt
import importlib.metadata as md
import platform

import torch

for package in ["uni2ts", "torch", "lightning", "gluonts", "numpy", "pandas", "hydra-core"]:
    print(package, md.version(package))
assert md.version("uni2ts") == "2.0.0"
assert torch.cuda.is_available(), "Select a GPU runtime"
print(platform.python_version(), torch.cuda.get_device_name(0), torch.version.cuda)

In [ ]:
import hashlib
import urllib.request

ETT_COMMIT = "1d16c8f4f943005d613b5bc962e9eeb06058cf07"
expected = {
    "ETTh1.csv": "f18de3ad269cef59bb07b5438d79bb3042d3be49bdeecf01c1cd6d29695ee066",
    "ETTh2.csv": "a3dc2c597b9218c7ce1cd55eb77b283fd459a1d09d753063f944967dd6b9218b",
    "ETTm1.csv": "6ce1759b1a18e3328421d5d75fadcb316c449fcd7cec32820c8dafda71986c9e",
    "ETTm2.csv": "db973ca252c6410a30d0469b13d696cf919648d0f3fd588c60f03fdbdbadd1fd",
}
data_dir = Path("/content/official-ett")
data_dir.mkdir(exist_ok=True)
for name, sha in expected.items():
    group = "ETT-small"
    path = data_dir / name
    urllib.request.urlretrieve(
        f"https://raw.githubusercontent.com/zhouhaoyi/ETDataset/{ETT_COMMIT}/{group}/{name}", path
    )
    assert hashlib.sha256(path.read_bytes()).hexdigest() == sha
print("four official ETT fingerprints verified")

In [ ]:
from uni2ts.model.moirai import MoiraiForecast, MoiraiModule

REPO = "Salesforce/moirai-1.1-R-small"
REVISION = "0c24ab99db2c1a70ea2a0fc03bf113329772ac64"
module = MoiraiModule.from_pretrained(
    REPO, revision=REVISION, cache_dir="/content/hf-moirai1"
).cuda()
model = MoiraiForecast(
    prediction_length=96,
    target_dim=7,
    feat_dynamic_real_dim=0,
    past_feat_dynamic_real_dim=0,
    context_length=512,
    module=module,
    patch_size=64,
    num_samples=8,
).eval()
x = torch.randn(1, 512, 7, device="cuda")
with torch.no_grad():
    y = model(
        x,
        torch.ones_like(x, dtype=torch.bool),
        torch.zeros(1, 512, dtype=torch.bool, device="cuda"),
    )
assert y.shape == (1, 8, 96, 7) and torch.isfinite(y).all()
print("synthetic joint multivariate shape", tuple(y.shape))
del model, module, x, y
torch.cuda.empty_cache()

In [ ]:
import json

RESULT = Path("results/raw/moirai1_gpu_probe.json")
subprocess.run(
    [
        "python",
        "-B",
        "scripts/probe_moirai1_cpu.py",
        "--data",
        str(data_dir / "ETTh1.csv"),
        "--cache-dir",
        "/content/hf-moirai1",
        "--output",
        str(RESULT),
        "--device",
        "cuda",
    ],
    check=True,
)
gpu = json.loads(RESULT.read_text())
assert gpu["revision"] == REVISION and gpu["gpu_validated"]
assert all(item["direct_output"] and item["finite"] for item in gpu["horizons"])
assert gpu["finetune"]["optimizer_steps"] in (1, 2)
assert gpu["finetune"]["parameter_updated"] and gpu["finetune"]["checkpoint_round_trip"]
assert gpu["cuda"]["peak_gpu_memory_mb"] > 0
print(json.dumps(gpu["cuda"], indent=2))

In [ ]:
# Review this untracked GPU evidence before updating the committed selection manifest.
for path in [
    ".cache/",
    "checkpoints/example.ckpt",
    "data/ETTh1.csv",
    "results/raw/moirai1_gpu_probe.json",
]:
    check = subprocess.run(["git", "check-ignore", "-q", path])
    assert check.returncode == 0, f"not ignored: {path}"
tracked_large = subprocess.check_output(["git", "ls-files"], text=True).splitlines()
assert not any(p.endswith((".ckpt", ".pt", ".pth", ".bin", ".safetensors")) for p in tracked_large)
print("cache, checkpoint, raw data, probe result, and weights are excluded from Git")